# CGW and chunked SGWB explorer

Run the cells from top to bottom. The notebook uses the same population generator and injection code as the project pipeline. The chunked workflow evaluates the cumulative SGWB optimal-statistic S/N one chunk at a time, saves only important binaries, and restores the live pulsars when it finishes.

The old distance-scaling consistency generator is intentionally not used here.


In [1]:
# 1. Configuration and project-root discovery
from pathlib import Path
import json
import sys
import gc
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "config.py").is_file() and (candidate / "SMBHB_pop_synth.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find the repository root containing config.py and "
        "SMBHB_pop_synth.py. Start Jupyter inside this repository."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Population and pulsar settings. These names are used by later cells.
POPULATION_CONFIG_NAME = "pessimistic"
PULSAR_SCENARIO = "baseline"
MIN_BASELINE_YEARS = 0.0
DEBUG_PULSAR_NAMES = None

# Chunked SGWB settings. Increase these only after a small run succeeds.
SGWB_TOTAL_BINARIES = 1_000_000
SGWB_CHUNK_SIZE = 1_000_000
SGWB_RANDOM_SEED = 26072001
SGWB_SNR_LOW = 3.0
SGWB_SNR_HIGH = 3.4
SGWB_TARGET_SNR = 3.2
SGWB_N_LOUDEST_PER_CHUNK = 500
SGWB_N_NEAREST_PER_CHUNK = 50
SGWB_N_EXTREMES_PER_CHUNK = 10
RUN_CHUNKED_SGWB = True

# Optional expensive CGW analysis for a retained subset.
RUN_CGW_SNR = True
N_CGW_CANDIDATES = 100
DEBUG_CGW = True
DEBUG_DENOMINATOR_THRESHOLD = 1e-3
DEBUG_RHO_SQ_THRESHOLD = 1.0

if SGWB_TOTAL_BINARIES <= 0 or SGWB_CHUNK_SIZE <= 0:
    raise ValueError("SGWB_TOTAL_BINARIES and SGWB_CHUNK_SIZE must be positive.")
if SGWB_TOTAL_BINARIES % SGWB_CHUNK_SIZE:
    raise ValueError("SGWB_TOTAL_BINARIES must be divisible by SGWB_CHUNK_SIZE.")
if not (0 < SGWB_SNR_LOW <= SGWB_SNR_HIGH):
    raise ValueError("Set a valid SGWB S/N interval.")

SGWB_N_CHUNKS = SGWB_TOTAL_BINARIES // SGWB_CHUNK_SIZE
SGWB_OUTPUT_DIR = PROJECT_ROOT / "debug" / "sgwb_chunked_population"
SGWB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Chunks: {SGWB_N_CHUNKS} x {SGWB_CHUNK_SIZE:,} binaries")


Project root: /Users/blaze/PhD_work/SMBHB_proj/SMBHB_EM_vs_GW/SMBHB_population_injection
Chunks: 1 x 1,000,000 binaries


In [2]:
# 2. Imports and project APIs
import config
import SMBHB_pop_synth as smbhb_module
from config import generate_population
from data_loader import load_pulsars, filter_pulsars_15yr, parse_pulsar_parameters
from pta_builder import build_pta_and_params
from signal_injection import inject_population_nufft, _get_psr_radec
from CGW_SNR import compute_cgw_snr_optimal_population_fast
from enterprise.pulsar import Pulsar as EnterprisePulsar
from enterprise_extensions.frequentist import optimal_statistic as opt_stat

if POPULATION_CONFIG_NAME not in config.POPULATION_CONFIGS:
    raise KeyError(
        f"Unknown population preset {POPULATION_CONFIG_NAME!r}; "
        f"choose from {sorted(config.POPULATION_CONFIGS)}"
    )

population_config = dict(config.POPULATION_CONFIGS[POPULATION_CONFIG_NAME])
print("Population preset:", POPULATION_CONFIG_NAME)
print("Noise file:", config.NOISEFILE)


Population preset: pessimistic
Noise file: /Users/blaze/PhD_work/SMBHB_proj/SMBHB_EM_vs_GW/SMBHB_population_injection/meerkat_45yr_noise.json


In [3]:
# 3. Load pulsars and define the authoritative time span
psrs = load_pulsars(verbose=True, scenario=PULSAR_SCENARIO)
psrs, _, Tspan_seconds = filter_pulsars_15yr(
    psrs,
    min_baseline_years=MIN_BASELINE_YEARS,
    verbose=True,
)

if DEBUG_PULSAR_NAMES is not None:
    wanted = set(DEBUG_PULSAR_NAMES)
    psrs = [psr for psr in psrs if psr.name in wanted]

if not psrs:
    raise RuntimeError("No pulsars remain after filtering.")
if Tspan_seconds <= 0:
    raise RuntimeError(f"Invalid Tspan_seconds={Tspan_seconds!r}")

# Only TOAs are snapshotted here: libstempo residuals/toas may be methods,
# and the injection path mutates stoas, which is the state we restore.
original_stoas = {
    psr.name: np.array(psr.stoas, dtype=float, copy=True)
    for psr in psrs
}
raw_noise_params = json.loads(Path(config.NOISEFILE).read_text())
parsed_noise_params = parse_pulsar_parameters(config.NOISEFILE)

print(f"Using {len(psrs)} pulsars")
print(f"Tspan = {Tspan_seconds / (365.25 * 86400.0):.4f} yr")


LOADING NANOGRAV PULSARS (libstempo)  —  scenario: baseline
Found 82 .par files → 82 unique pulsars
[tempo2Util.C:396] Warning: [TIM1] Please place MODE flags in the parameter file 
✓ Loaded J0030+0451 (2880 TOAs) [tim=J0030+0451]
[tempo2Util.C:401] Warning: [DUP1] duplicated warnings have been suppressed.
✓ Loaded J0101-6422 (1430 TOAs) [tim=J0101-6422]
✓ Loaded J0125-2327 (3170 TOAs) [tim=J0125-2327]
✓ Loaded J0437-4715 (3517 TOAs) [tim=J0437-4715]
✓ Loaded J0610-2100 (2758 TOAs) [tim=J0610-2100]
✓ Loaded J0613-0200 (3067 TOAs) [tim=J0613-0200]
✓ Loaded J0614-3329 (3499 TOAs) [tim=J0614-3329]
✓ Loaded J0636-3044 (2217 TOAs) [tim=J0636-3044]
✓ Loaded J0711-6830 (2830 TOAs) [tim=J0711-6830]
✓ Loaded J0900-3144 (3262 TOAs) [tim=J0900-3144]
✓ Loaded J0931-1902 (1903 TOAs) [tim=J0931-1902]
✓ Loaded J0955-6150 (6167 TOAs) [tim=J0955-6150]
✓ Loaded J1012-4235 (2898 TOAs) [tim=J1012-4235]
✓ Loaded J1017-7156 (3321 TOAs) [tim=J1017-7156]
✓ Loaded J1022+1001 (2945 TOAs) [tim=J1022+1001]
✓ Load

In [4]:
# 4. Pulsar inspection helpers
def _array_attribute(obj, name):
    value = getattr(obj, name)
    return value() if callable(value) else value

psr_rows = []
for psr in psrs:
    toas = np.asarray(_array_attribute(psr, "toas"), dtype=float)
    ra_rad, dec_rad = _get_psr_radec(psr)
    psr_rows.append({
        "name": psr.name,
        "n_toa": len(toas),
        "ra_rad": float(ra_rad),
        "dec_rad": float(dec_rad),
        "ra_deg": float(np.degrees(ra_rad)),
        "dec_deg": float(np.degrees(dec_rad)),
        "toa_min_yr": float(np.min(toas) / (365.25 * 86400.0)),
        "toa_max_yr": float(np.max(toas) / (365.25 * 86400.0)),
    })
psr_table = pd.DataFrame(psr_rows).sort_values("name").reset_index(drop=True)
display(psr_table)


,name,n_toa,ra_rad,dec_rad,ra_deg,dec_deg,toa_min_yr,toa_max_yr
0,J0030+0451,2880,0.132894,0.084841,7.614265,4.861031,0.001863,0.001906
1,J0101-6422,1430,0.266971,-1.123557,15.296317,-64.375048,0.001862,0.001906
2,J0125-2327,3170,0.370961,-0.409319,21.254476,-23.452260,0.001857,0.001906
3,J0437-4715,3517,1.209799,-0.824713,69.316369,-47.252584,0.001859,0.001906
4,J0610-2100,2758,1.615419,-0.366654,92.556678,-21.007708,0.001862,0.001906
...,...,...,...,...,...,...,...,...
77,J2236-5527,1599,5.920437,-0.968022,339.216041,-55.463583,0.001857,0.001906
78,J2241-5236,3405,5.941538,-0.918219,340.425071,-52.610059,0.001855,0.001906
79,J2317+1439,1948,6.096234,0.255842,349.288482,14.658693,0.001863,0.001906
80,J2322+2057,1756,6.119002,0.365659,350.593002,20.950696,0.001863,0.001906


In [5]:
# 5. Generate one ordinary in-memory population for basic inspection
population, strain_data = generate_population(
    population_config,
    smbhb_module,
    T_obs_seconds=Tspan_seconds,
    n_binaries=min(population_config["n_binaries"], SGWB_CHUNK_SIZE),
    compute_strain=True,
    seed=SGWB_RANDOM_SEED,
)

print(f"Generated {len(population):,} binaries")
print(f"Population memory: {population.memory_gb():.4f} GB")

def population_table(pop, indices=None):
    if indices is None:
        indices = np.arange(len(pop))
    indices = np.asarray(indices, dtype=int)
    fields = ("f", "Mc", "Mtot", "D_comov", "z", "h0", "ecc",
              "ra", "dec", "psi", "iota", "phi0", "cgw_snr")
    return pd.DataFrame({
        field: np.asarray(getattr(pop, field))[indices]
        for field in fields
    }, index=indices)

display(population_table(population, np.arange(min(10, len(population)))))


Generated 1,000,000 binaries
Population memory: 0.0894 GB


/opt/homebrew/anaconda3/envs/enter312/lib/python3.12/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast


,f,Mc,Mtot,D_comov,z,h0,ecc,ra,dec,psi,iota,phi0,cgw_snr
0,3.064021e-09,8.781835e+07,2.017644e+08,4199.139750,1.321886,9.077577e-19,0.016349,6.002385,-0.416897,1.548253,0.439257,5.197430,0.0
1,2.115551e-09,1.103162e+08,2.585250e+08,2816.482223,0.766757,1.288720e-18,0.100588,4.075323,-1.030073,0.413069,2.270503,1.255167,0.0
2,3.523284e-09,2.614210e+07,7.961161e+07,2788.220438,0.757093,1.653744e-19,0.028953,5.615433,0.048207,0.035236,0.599791,3.111247,0.0
3,5.559740e-09,3.195487e+07,7.353648e+07,4690.860808,1.569530,2.398729e-19,0.011799,3.401846,0.871186,0.332795,0.693709,4.266556,0.0
4,3.010599e-09,1.806480e+08,4.164849e+08,2152.612288,0.553313,4.454140e-18,0.106971,4.225352,-0.232112,2.067229,1.183767,2.212718,0.0
5,2.493535e-09,1.488981e+08,3.432172e+08,4321.344382,1.380454,1.884719e-18,0.873455,0.822344,-0.448779,2.954293,1.301499,5.697425,0.0
6,4.026250e-09,9.589668e+07,2.550626e+08,2054.388714,0.523921,1.946698e-18,0.153846,0.607560,-0.279288,2.961753,1.167779,1.086327,0.0
7,2.417160e-09,2.228398e+07,1.788171e+08,1121.151659,0.267285,1.971711e-19,0.060223,2.831634,-0.262977,0.654622,1.250214,3.579076,0.0
8,2.520067e-09,3.520066e+07,9.129308e+07,4129.345657,1.289254,1.749205e-19,0.128801,0.459390,-1.071699,0.620151,0.192980,3.210057,0.0
9,2.751980e-09,5.524137e+07,1.330692e+08,2902.372637,0.796473,4.758440e-19,0.059740,1.337664,0.153085,1.592538,1.837832,4.216988,0.0


In [6]:
# 6. Chunked SGWB helpers
SOURCE_FIELDS = (
    "f", "Mc", "Mtot", "D_comov", "z", "h0", "ecc",
    "ra", "dec", "psi", "iota", "phi0",
)

def _extreme_indices(values, n, largest):
    values = np.asarray(values, dtype=float)
    valid = np.flatnonzero(np.isfinite(values))
    if not len(valid):
        return np.array([], dtype=int)
    n = min(int(n), len(valid))
    order = np.argsort(values[valid])
    return valid[order[-n:] if largest else order[:n]]

def select_important_indices(pop):
    proxy = np.asarray(pop.h0, dtype=float) / (2.0 * np.pi * np.asarray(pop.f, dtype=float))
    selected = set(_extreme_indices(proxy, SGWB_N_LOUDEST_PER_CHUNK, True).tolist())
    selected.update(_extreme_indices(np.asarray(pop.D_comov), SGWB_N_NEAREST_PER_CHUNK, False).tolist())
    for field in ("f", "Mc", "Mtot", "D_comov", "z", "h0", "ecc"):
        selected.update(_extreme_indices(getattr(pop, field), SGWB_N_EXTREMES_PER_CHUNK, False).tolist())
        selected.update(_extreme_indices(getattr(pop, field), SGWB_N_EXTREMES_PER_CHUNK, True).tolist())
    return np.asarray(sorted(selected), dtype=int)

def compact_sources(pop, indices, chunk_id, seed):
    result = {
        field: np.asarray(getattr(pop, field))[indices].copy()
        for field in SOURCE_FIELDS
    }
    result.update({
        "local_index": np.asarray(indices, dtype=np.int64),
        "chunk_id": np.full(len(indices), chunk_id, dtype=np.int64),
        "chunk_seed": np.full(len(indices), seed, dtype=np.int64),
    })
    return result

def restore_pulsars():
    for psr in psrs:
        psr.stoas[:] = original_stoas[psr.name]

def sgwb_snr_from_current_stoas():
    enterprise_psrs = [
        EnterprisePulsar(psr, ephem="DE440", backend="tempo2")
        for psr in psrs
    ]
    pta, _, pta_params = build_pta_and_params(
        psrs=enterprise_psrs,
        noise_params=raw_noise_params,
        Tspan=Tspan_seconds,
        include_GW=True, include_RN=True, include_WN=True,
    )
    statistic = opt_stat.OptimalStatistic(enterprise_psrs, pta=pta, orf="hd")
    _, _, _, os_value, os_sigma = statistic.compute_os(params=pta_params)
    snr = float(os_value / os_sigma)
    del statistic, pta, enterprise_psrs
    gc.collect()
    return snr


In [7]:
# 7. Run the chunked HPC-style SGWB workflow
def run_chunked_sgwb():
    run_dir = SGWB_OUTPUT_DIR / f"seed_{SGWB_RANDOM_SEED}_{SGWB_TOTAL_BINARIES}binaries"
    source_dir = run_dir / "important_sources"
    run_dir.mkdir(parents=True, exist_ok=True)
    source_dir.mkdir(exist_ok=True)

    restore_pulsars()
    records = []
    cumulative_edges = None
    cumulative_centres = None
    cumulative_h_squared = None

    for chunk_id in range(SGWB_N_CHUNKS):
        seed = SGWB_RANDOM_SEED + chunk_id
        pop, chunk_strain = generate_population(
            population_config, smbhb_module,
            T_obs_seconds=Tspan_seconds,
            n_binaries=SGWB_CHUNK_SIZE,
            compute_strain=True, seed=seed,
        )

        edges = np.asarray(chunk_strain["bin_edges"], dtype=float)
        centres = np.asarray(chunk_strain["bin_centres"], dtype=float)
        bin_assignment = np.asarray(chunk_strain["bin_assignment"], dtype=int)
        h_square_individual = np.asarray(
            chunk_strain["h_square_individual"], dtype=float
        )
        in_range = (bin_assignment >= 0) & (bin_assignment < len(centres))
        h_squared = np.bincount(
            bin_assignment[in_range],
            weights=h_square_individual[in_range],
            minlength=len(centres),
        )
        h_squared = h_squared[:len(centres)]

        if cumulative_h_squared is None:
            cumulative_edges = edges
            cumulative_centres = centres
            cumulative_h_squared = np.zeros_like(h_squared, dtype=float)
        elif not np.array_equal(cumulative_edges, edges):
            raise RuntimeError("Inconsistent strain-bin grids between chunks.")
        cumulative_h_squared += h_squared

        important = select_important_indices(pop)
        np.savez_compressed(
            source_dir / f"chunk_{chunk_id:04d}.npz",
            **compact_sources(pop, important, chunk_id, seed),
        )

        # inject_population_nufft adds this chunk to the current stoas; no
        # reset occurs inside the loop, so the signal remains cumulative.
        inject_population_nufft(
            psrs, pop, verbose=False, eps=1e-6,
            cache_precomputed_amplitudes=True,
        )
        snr = sgwb_snr_from_current_stoas()
        total = (chunk_id + 1) * SGWB_CHUNK_SIZE
        records.append({
            "chunk_id": chunk_id,
            "seed": seed,
            "total_generated": total,
            "important_saved": len(important),
            "cumulative_sgwb_snr": snr,
        })
        print(f"chunk {chunk_id + 1}/{SGWB_N_CHUNKS}: cumulative S/N={snr:.5f}")

        del pop, chunk_strain, important, h_squared
        gc.collect()

        if snr >= SGWB_SNR_LOW:
            break

    h_c_total = np.sqrt(
        cumulative_h_squared * cumulative_centres / np.diff(cumulative_edges)
    )
    np.savez_compressed(
        run_dir / "cumulative_background.npz",
        bin_edges_hz=cumulative_edges,
        bin_centres_hz=cumulative_centres,
        h_squared_per_bin=cumulative_h_squared,
        h_c_total=h_c_total,
    )
    manifest = {
        "Tspan_seconds": float(Tspan_seconds),
        "chunk_size": SGWB_CHUNK_SIZE,
        "chunks_processed": len(records),
        "records": records,
        "final_sgwb_snr": records[-1]["cumulative_sgwb_snr"],
        "source_directory": str(source_dir),
    }
    (run_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    restore_pulsars()
    return {"run_dir": run_dir, "manifest": manifest}

chunked_sgwb_result = run_chunked_sgwb() if RUN_CHUNKED_SGWB else None
if chunked_sgwb_result is None:
    print("Skipped chunked SGWB run; set RUN_CHUNKED_SGWB=True after the setup cells succeed.")


chunk 1/1: cumulative S/N=2.99516


In [8]:
# 8. Inspect the chunked run and load only retained sources
if chunked_sgwb_result is not None:
    manifest = chunked_sgwb_result["manifest"]
    chunk_history = pd.DataFrame(manifest["records"])
    display(chunk_history)

    source_files = sorted(Path(manifest["source_directory"]).glob("chunk_*.npz"))
    retained_rows = []
    for source_file in source_files:
        data = np.load(source_file)
        n = len(data["f"])
        for i in range(n):
            row = {field: float(data[field][i]) for field in SOURCE_FIELDS}
            row.update({
                "chunk_id": int(data["chunk_id"][i]),
                "local_index": int(data["local_index"][i]),
            })
            retained_rows.append(row)
    important_source_table = pd.DataFrame(retained_rows)
    important_source_table["residual_proxy_s"] = (
        important_source_table["h0"] / (2.0 * np.pi * important_source_table["f"])
    )
    important_source_table = important_source_table.sort_values(
        "residual_proxy_s", ascending=False
    ).reset_index(drop=True)
    print(f"Retained {len(important_source_table):,} important sources.")
    display(important_source_table.head(20))
else:
    important_source_table = pd.DataFrame()


,chunk_id,seed,total_generated,important_saved,cumulative_sgwb_snr
0,0,26072001,1000000,620,2.995163


Retained 620 important sources.


,f,Mc,Mtot,D_comov,z,h0,ecc,ra,dec,psi,iota,phi0,chunk_id,local_index,residual_proxy_s
0,2.005860e-09,3.314106e+09,7.675230e+09,1228.710682,0.295016,6.728909e-16,0.500197,2.186742,0.022161,0.905026,1.773390,1.465454,0,656003,5.339052e-08
1,2.226161e-09,2.246640e+09,5.165403e+09,591.830135,0.136569,7.181140e-16,0.289622,2.804163,0.987876,1.435913,0.403757,6.098891,0,134539,5.134013e-08
2,2.529496e-09,1.916761e+09,4.421385e+09,459.772129,0.105299,7.582512e-16,0.150501,3.552963,-0.326717,0.830746,2.969578,4.785904,0,771673,4.770888e-08
3,4.520328e-09,1.730657e+09,4.041193e+09,338.438764,0.076992,1.257553e-15,0.121523,1.628315,-0.480080,2.629655,1.987417,4.643077,0,757426,4.427684e-08
4,2.145172e-09,1.655127e+09,3.839503e+09,498.568242,0.114435,4.932553e-16,0.927296,3.816197,0.542547,1.472077,2.864757,5.070506,0,69151,3.659568e-08
5,3.018627e-09,1.452062e+09,3.337571e+09,405.787934,0.092656,6.038620e-16,0.011105,2.932841,-0.701405,2.557786,1.981936,2.784411,0,132226,3.183819e-08
6,2.381628e-09,1.495174e+09,3.481539e+09,478.813006,0.109777,4.635821e-16,0.022923,1.880131,-0.028264,2.434512,0.209174,4.651781,0,294016,3.097939e-08
7,3.882350e-09,2.114953e+09,5.247048e+09,800.426721,0.187005,7.160515e-16,0.034283,3.727643,-0.314831,2.355300,1.252191,1.810488,0,242881,2.935416e-08
8,2.856890e-09,2.152958e+09,5.067403e+09,960.929998,0.226745,5.119164e-16,0.090958,5.224420,0.978454,1.342150,2.233216,1.224409,0,213809,2.851843e-08
9,2.437776e-09,2.009093e+09,4.683685e+09,900.125398,0.211591,4.345069e-16,0.547841,6.252999,0.253329,1.022775,2.105869,6.010292,0,795307,2.836763e-08


In [9]:
# 9. Optional CGW S/N for only the retained loud candidates
if RUN_CGW_SNR and not important_source_table.empty:
    candidate_table = important_source_table.head(N_CGW_CANDIDATES).copy()
    candidate_binaries = [
        SimpleNamespace(**{field: float(row[field]) for field in SOURCE_FIELDS}, cgw_snr=0.0)
        for _, row in candidate_table.iterrows()
    ]
    enterprise_psrs = [EnterprisePulsar(psr, ephem="DE440", backend="tempo2") for psr in psrs]
    pta, _, _ = build_pta_and_params(
        psrs=enterprise_psrs, noise_params=raw_noise_params, Tspan=Tspan_seconds,
        include_GW=True, include_RN=True, include_WN=True,
    )
    candidate_snrs, candidate_breakdowns = compute_cgw_snr_optimal_population_fast(
        psrs=enterprise_psrs, pta=pta, population=candidate_binaries,
        raw_noise_params=raw_noise_params, parsed_noise_params=parsed_noise_params,
        Tspan=Tspan_seconds, profile=True, return_breakdown=True,
        debug=DEBUG_CGW, debug_denominator_threshold=DEBUG_DENOMINATOR_THRESHOLD,
        debug_rho_sq_threshold=DEBUG_RHO_SQ_THRESHOLD,
    )
    candidate_table["recomputed_cgw_snr"] = np.asarray(candidate_snrs, dtype=float)
    display(candidate_table.sort_values("recomputed_cgw_snr", ascending=False).head(20))
else:
    candidate_snrs, candidate_breakdowns = None, None
    print("Skipped retained-source CGW S/N.")


Precompute time: 3.27 s

CGW debug selection: 10 source(s): 10 top-10, 0 single-pulsar dominated, 0 non-finite.

CGW debug source 3
Reason: top-10 by CW S/N (rank 3, rho=1.361132e-01)
  source.f        = 4.5203279512799185e-09
  source.Mc       = 1730657482.1348753
  source.Mtot     = 4041192529.5942612
  source.D_comov  = 338.43876385776042
  source.z        = 0.07699157095997522
  source.h0       = 1.2575532626176855e-15
  source.ecc      = 0.1215231894636874
  source.ra       = 1.6283148080071743
  source.dec      = -0.48008046700048912
  source.psi      = 2.6296546484246774
  source.iota     = 1.9874168637774652
  source.phi0     = 4.6430768458028249
[CGW debug] source=3 final rho^2=1.8526792991350075e-02, rho=1.3611316244709795e-01

CGW debug source 7
Reason: top-10 by CW S/N (rank 10, rho=6.707455e-02)
  source.f        = 3.8823500318619826e-09
  source.Mc       = 2114953083.2888708
  source.Mtot     = 5247048129.1776762
  source.D_comov  = 800.42672088782501
  source.z        = 

,f,Mc,Mtot,D_comov,z,h0,ecc,ra,dec,psi,iota,phi0,chunk_id,local_index,residual_proxy_s,recomputed_cgw_snr
91,1.162666e-08,7.414775e+08,1.705668e+09,193.760723,0.043738,9.832446e-16,0.091553,3.667288,-0.877656,1.975889,2.880014,4.902640,0,494404,1.345943e-08,0.808015
58,6.743207e-09,2.089638e+09,5.625415e+09,1347.420284,0.326129,6.486092e-16,0.001512,2.170671,0.299033,2.321644,3.089049,3.717827,0,751839,1.530865e-08,0.169848
3,4.520328e-09,1.730657e+09,4.041193e+09,338.438764,0.076992,1.257553e-15,0.121523,1.628315,-0.480080,2.629655,1.987417,4.643077,0,757426,4.427684e-08,0.136113
18,4.368404e-09,1.233995e+09,3.041780e+09,364.128970,0.082952,6.525618e-16,0.159243,5.625643,0.059934,1.396454,0.403117,3.001480,0,705979,2.377491e-08,0.129465
84,4.867565e-09,1.325318e+09,3.235526e+09,719.292086,0.167230,4.204103e-16,0.007529,5.032905,-0.971167,0.325353,0.668811,0.732283,0,514508,1.374617e-08,0.114575
63,5.881362e-09,1.003106e+09,2.345171e+09,368.493825,0.083967,5.570274e-16,0.076960,5.196352,0.248656,2.380197,2.187394,2.406117,0,809454,1.507366e-08,0.109538
28,4.250367e-09,2.623358e+09,6.053474e+09,2008.267397,0.510295,5.097216e-16,0.814915,3.624020,-1.317048,1.714779,2.358457,4.230965,0,209251,1.908652e-08,0.097110
32,4.512953e-09,2.870954e+09,6.971884e+09,2676.112316,0.719299,5.044452e-16,0.122892,1.647250,-0.901029,2.371184,2.375510,1.109703,0,650271,1.778989e-08,0.086148
70,3.885330e-09,2.861043e+09,6.573347e+09,4286.088691,1.363366,3.503633e-16,0.021737,5.747365,-0.658114,0.111361,2.758448,0.140300,0,847151,1.435195e-08,0.067102
7,3.882350e-09,2.114953e+09,5.247048e+09,800.426721,0.187005,7.160515e-16,0.034283,3.727643,-0.314831,2.355300,1.252191,1.810488,0,242881,2.935416e-08,0.067075


In [10]:
# ============================================================================
# GW-amplitude sensitivity test for the retained-candidate CGW S/N.
#
# PART 1 (functions): paste once, anywhere above step 9b.
# PART 2 (cell "9b"): paste right after your step 9 cell.
#
# Answers: "does building the PTA with the MPTA common-process amplitude instead
# of the builder default (log10A=-14.0) change the CGW S/N of my candidates enough
# to matter (values, ranking, which pulsar dominates)?"
# Needs only: numpy, pandas, scipy, and your own build_pta_and_params +
# compute_cgw_snr_optimal_population_fast.
# ============================================================================
import time
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# MPTA 4.5-yr common process (Miles et al. 2025): fixed gamma=13/3 -> log10A = -14.28 +/- 0.21;
# free fit -> log10A = -14.25, gamma = 3.60.  Edit/remove entries freely; the FIRST entry is the reference.
GW_TEST_CONFIGS = {
    "default_-14.00":         dict(gw_log10_A=-14.00, gw_gamma=13.0 / 3.0),
    "MPTA_-14.28":            dict(gw_log10_A=-14.28, gw_gamma=13.0 / 3.0),
    "MPTA_-14.07(+1sig)":     dict(gw_log10_A=-14.07, gw_gamma=13.0 / 3.0),
    "MPTA_-14.49(-1sig)":     dict(gw_log10_A=-14.49, gw_gamma=13.0 / 3.0),
    "MPTA_free_-14.25_g3.6":  dict(gw_log10_A=-14.25, gw_gamma=3.60),
    "no_GWB(upper_bound)":    dict(include_GW=False),
}


def _gw_constants_in_pta(pta, crn_name="gw"):
    """Return {param_name: value} for the common-process Constants actually baked into the PTA."""
    out = {}
    for sc in pta._signalcollections:
        for s in sc._signals:
            for p in s._params.values():
                nm = getattr(p, "name", "")
                if nm.endswith(f"_{crn_name}_log10_A") or nm.endswith(f"_{crn_name}_gamma"):
                    out[nm] = getattr(p, "value", None)
    return out


def _check_applied(pta, kw, crn_name, label):
    """Verify the amplitude/index that ended up in the PTA is the one requested
    (guards against set_default_params overriding the Constant via a matching noise-dict key)."""
    want_A = kw.get("gw_log10_A", -14.0) if kw.get("include_GW", True) else -20.0
    want_g = kw.get("gw_gamma", 13.0 / 3.0)
    try:
        got = _gw_constants_in_pta(pta, crn_name)
        As = np.array([v for k, v in got.items() if k.endswith("_log10_A")], float)
        Gs = np.array([v for k, v in got.items() if k.endswith("_gamma")], float)
        ok = (As.size > 0 and np.allclose(As, want_A) and np.allclose(Gs, want_g))
        note = (f"applied log10A in [{As.min():.2f}, {As.max():.2f}], gamma in "
                f"[{Gs.min():.2f}, {Gs.max():.2f}] (wanted {want_A:.2f}, {want_g:.2f})") if As.size else "no GW constants found"
    except Exception as e:                                   # structure differs: don't fail the test
        return None, f"could not inspect PTA ({type(e).__name__})"
    if not ok:
        print(f"  [WARNING] config {label!r}: {note}. A noise-dict key named '<psr>_{crn_name}_log10_A' "
              f"(or crn_name mismatch) is probably overriding the requested amplitude.")
    return ok, note


def run_gw_amplitude_comparison(psrs, population, raw_noise_params, parsed_noise_params, Tspan,
                                build_fn, compute_fn, configs=None, reference=None,
                                build_kwargs=None, reuse=None, top_k=20, tol=0.05,
                                crn_name="gw", verbose=True):
    """
    Build one PTA per config, compute the CGW S/N of `population` under each, and compare.

    reuse : optional {config_name: (rho_array, breakdowns)} to skip recomputing a config you
            already have (e.g. {"default_-14.00": (candidate_snrs, candidate_breakdowns)} from step 9).
    top_k : size of the "top candidates" used for ranking overlap and for the verdict.
    tol   : fractional change in rho (within the reference top-k) above which a rerun is flagged.
    Returns dict(table, summary, verdict, rho, dominant, applied).
    """
    configs = dict(configs or GW_TEST_CONFIGS)
    reference = reference or next(iter(configs))
    reuse = reuse or {}
    rho, dom, applied = {}, {}, {}

    for name, cfg in configs.items():
        if name in reuse:
            r, bd = reuse[name]
            applied[name] = (None, "reused from earlier run (not re-verified)")
            if verbose:
                print(f"[{name}] reusing existing result")
        else:
            t0 = time.time()
            kw = dict(include_GW=True, include_RN=True, include_WN=True)
            kw.update(build_kwargs or {})
            kw.update(cfg)
            pta, _, _ = build_fn(psrs=psrs, noise_params=raw_noise_params, Tspan=Tspan, **kw)
            applied[name] = _check_applied(pta, kw, crn_name, name)
            r, bd = compute_fn(psrs=psrs, pta=pta, population=population,
                               raw_noise_params=raw_noise_params,
                               parsed_noise_params=parsed_noise_params, Tspan=Tspan,
                               return_breakdown=True, debug=False)
            del pta
            if verbose:
                print(f"[{name}] done in {time.time() - t0:.1f} s  | {applied[name][1]}")
        rho[name] = np.asarray(r, dtype=float)
        dom[name] = [max(b, key=b.get) if b else None for b in bd]

    ref = rho[reference]
    k = int(min(top_k, len(ref)))
    order_ref = np.argsort(-ref)
    top_ref = set(order_ref[:k].tolist())

    table = pd.DataFrame({f"rho[{n}]": v for n, v in rho.items()})
    rows = []
    for n, r in rho.items():
        if n == reference:
            continue
        ratio = np.divide(r, ref, out=np.full_like(r, np.nan), where=ref > 0)
        dev = np.abs(ratio - 1.0)
        top_n = set(np.argsort(-r)[:k].tolist())
        in_top = np.array(sorted(top_ref), dtype=int)
        fin, fin_top = np.isfinite(ratio), np.isfinite(ratio[in_top])
        sp = np.nan
        if len(ref) > 1 and np.ptp(ref) > 0 and np.ptp(r) > 0:
            sp = spearmanr(ref, r)
            sp = getattr(sp, "statistic", sp[0])
        rows.append(dict(
            config=n, applied_ok=applied[n][0],
            median_change_pct=100 * (np.median(ratio[fin]) - 1) if fin.any() else np.nan,
            p16_pct=100 * (np.percentile(ratio[fin], 16) - 1) if fin.any() else np.nan,
            p84_pct=100 * (np.percentile(ratio[fin], 84) - 1) if fin.any() else np.nan,
            max_abs_change_pct=100 * np.max(dev[fin]) if fin.any() else np.nan,
            max_abs_change_topk_pct=100 * np.max(dev[in_top][fin_top]) if fin_top.any() else np.nan,
            worst_source_in_topk=int(in_top[fin_top][np.argmax(dev[in_top][fin_top])]) if fin_top.any() else -1,
            top1_same=bool(np.argmax(r) == np.argmax(ref)),
            topk_overlap=len(top_ref & top_n),
            spearman=float(sp),
            dominant_pulsar_changed_frac=float(np.mean([a != b for a, b in zip(dom[reference], dom[n])])),
        ))
    summary = pd.DataFrame(rows).set_index("config") if rows else pd.DataFrame()

    # ---- verdict, based on the first non-reference config that is a plain MPTA amplitude if present ----
    verdict = "Only one config was run; nothing to compare."
    if len(summary):
        primary = next((c for c in summary.index if c.startswith("MPTA_-14.28")), summary.index[0])
        s = summary.loc[primary]
        if not np.isfinite(s.max_abs_change_topk_pct):
            verdict = (f"{primary} vs {reference}: cannot be assessed (reference rho is zero/non-finite "
                       f"for the top candidates). Check the CGW S/N inputs.")
            if verbose:
                print("\n" + "=" * 78 + "\n" + verdict + "\n" + "=" * 78)
            return dict(table=table, summary=summary, verdict=verdict, rho=rho, dominant=dom, applied=applied)
        material = (s.max_abs_change_topk_pct > 100 * tol) or (not s.top1_same) or (s.topk_overlap < k)
        verdict = (
            f"{primary} vs {reference}: median change {s.median_change_pct:+.1f}% "
            f"(16-84%: {s.p16_pct:+.1f}% to {s.p84_pct:+.1f}%), max {s.max_abs_change_pct:.1f}% overall and "
            f"{s.max_abs_change_topk_pct:.1f}% within the reference top-{k} (source {s.worst_source_in_topk}); "
            f"top-1 {'unchanged' if s.top1_same else 'CHANGED'}, top-{k} overlap {int(s.topk_overlap)}/{k}, "
            f"Spearman {s.spearman:.4f}.\n"
            + ("=> MATERIAL at your tolerance (%.0f%%): recompute the CGW S/N with the new amplitude. "
               "If candidate retention (earlier steps) does not use the PTA, this only means rerunning step 9, "
               "not the simulations." % (100 * tol)
               if material else
               "=> Within tolerance (%.0f%%) for values, top-1 and top-%d membership: the default amplitude "
               "does not change your conclusions for these candidates." % (100 * tol, k))
        )
    if verbose:
        print("\n" + "=" * 78 + "\n" + verdict + "\n" + "=" * 78)
    return dict(table=table, summary=summary, verdict=verdict, rho=rho, dominant=dom, applied=applied)


# ============================================================================
# PART 2 - notebook cell "9b": paste right after step 9
# ============================================================================
RUN_GW_AMPLITUDE_TEST = True      # set False to skip
GW_TEST_TOP_K = 20
GW_TEST_TOL = 0.05                # 5% change in rho within the top-k counts as "material"

if RUN_CGW_SNR and RUN_GW_AMPLITUDE_TEST and not important_source_table.empty and candidate_snrs is not None:
    gw_cmp = run_gw_amplitude_comparison(
        psrs=enterprise_psrs,
        population=candidate_binaries,
        raw_noise_params=raw_noise_params,
        parsed_noise_params=parsed_noise_params,
        Tspan=Tspan_seconds,
        build_fn=build_pta_and_params,
        compute_fn=compute_cgw_snr_optimal_population_fast,
        # step 9 already computed the default (-14.0) result; reuse it. Remove this line if step 9's
        # build_pta_and_params call used a different gw_log10_A / gw_gamma / crn_name.
        reuse={"default_-14.00": (candidate_snrs, candidate_breakdowns)},
        top_k=GW_TEST_TOP_K, tol=GW_TEST_TOL,
    )
    display(gw_cmp["summary"].round(3))
    for _name, _rho in gw_cmp["rho"].items():
        candidate_table[f"cgw_snr[{_name}]"] = _rho
    display(candidate_table.sort_values("cgw_snr[default_-14.00]", ascending=False).head(20))
else:
    gw_cmp = None
    print("Skipped GW-amplitude comparison (needs step 9 to have run).")

[default_-14.00] reusing existing result
[MPTA_-14.28] done in 8.9 s  | applied log10A in [-14.28, -14.28], gamma in [4.33, 4.33] (wanted -14.28, 4.33)
[MPTA_-14.07(+1sig)] done in 8.6 s  | applied log10A in [-14.07, -14.07], gamma in [4.33, 4.33] (wanted -14.07, 4.33)
[MPTA_-14.49(-1sig)] done in 8.7 s  | applied log10A in [-14.49, -14.49], gamma in [4.33, 4.33] (wanted -14.49, 4.33)
[MPTA_free_-14.25_g3.6] done in 9.9 s  | applied log10A in [-14.25, -14.25], gamma in [3.60, 3.60] (wanted -14.25, 3.60)
[no_GWB(upper_bound)] done in 8.8 s  | applied log10A in [-20.00, -20.00], gamma in [4.33, 4.33] (wanted -20.00, 4.33)

MPTA_-14.28 vs default_-14.00: median change +48.8% (16-84%: +44.2% to +53.0%), max 57.5% overall and 53.7% within the reference top-20 (source 84); top-1 unchanged, top-20 overlap 20/20, Spearman 0.9987.
=> MATERIAL at your tolerance (5%): recompute the CGW S/N with the new amplitude. If candidate retention (earlier steps) does not use the PTA, this only means rerunni

,applied_ok,median_change_pct,p16_pct,p84_pct,max_abs_change_pct,max_abs_change_topk_pct,worst_source_in_topk,top1_same,topk_overlap,spearman,dominant_pulsar_changed_frac
config,,,,,,,,,,,
MPTA_-14.28,True,48.775,44.209,53.011,57.454,53.734,84,True,20,0.999,0.15
MPTA_-14.07(+1sig),True,11.373,10.641,12.010,12.593,12.136,84,True,20,1.000,0.02
MPTA_-14.49(-1sig),True,89.870,76.532,99.972,113.084,101.905,84,True,20,0.997,0.36
MPTA_free_-14.25_g3.6,True,86.421,73.959,96.569,109.244,99.094,84,True,20,0.997,0.34
no_GWB(upper_bound),True,511.896,234.436,818.978,1093.822,902.256,84,True,15,0.900,0.82


,f,Mc,Mtot,D_comov,z,h0,ecc,ra,dec,psi,...,chunk_id,local_index,residual_proxy_s,recomputed_cgw_snr,cgw_snr[default_-14.00],cgw_snr[MPTA_-14.28],cgw_snr[MPTA_-14.07(+1sig)],cgw_snr[MPTA_-14.49(-1sig)],cgw_snr[MPTA_free_-14.25_g3.6],cgw_snr[no_GWB(upper_bound)]
91,1.162666e-08,7.414775e+08,1.705668e+09,193.760723,0.043738,9.832446e-16,0.091553,3.667288,-0.877656,1.975889,...,0,494404,1.345943e-08,0.808015,0.808015,1.150823,0.890229,1.420632,1.357187,3.854046
58,6.743207e-09,2.089638e+09,5.625415e+09,1347.420284,0.326129,6.486092e-16,0.001512,2.170671,0.299033,2.321644,...,0,751839,1.530865e-08,0.169848,0.169848,0.245461,0.188322,0.300209,0.295598,0.650176
3,4.520328e-09,1.730657e+09,4.041193e+09,338.438764,0.076992,1.257553e-15,0.121523,1.628315,-0.480080,2.629655,...,0,757426,4.427684e-08,0.136113,0.136113,0.201661,0.151326,0.256845,0.253276,1.004672
18,4.368404e-09,1.233995e+09,3.041780e+09,364.128970,0.082952,6.525618e-16,0.159243,5.625643,0.059934,1.396454,...,0,705979,2.377491e-08,0.129465,0.129465,0.197794,0.145013,0.257665,0.253623,0.756810
84,4.867565e-09,1.325318e+09,3.235526e+09,719.292086,0.167230,4.204103e-16,0.007529,5.032905,-0.971167,0.325353,...,0,514508,1.374617e-08,0.114575,0.114575,0.176140,0.128480,0.231332,0.228111,1.148331
63,5.881362e-09,1.003106e+09,2.345171e+09,368.493825,0.083967,5.570274e-16,0.076960,5.196352,0.248656,2.380197,...,0,809454,1.507366e-08,0.109538,0.109538,0.165228,0.122146,0.215449,0.211346,0.823855
28,4.250367e-09,2.623358e+09,6.053474e+09,2008.267397,0.510295,5.097216e-16,0.814915,3.624020,-1.317048,1.714779,...,0,209251,1.908652e-08,0.097110,0.097110,0.145322,0.108227,0.186198,0.183426,0.508098
32,4.512953e-09,2.870954e+09,6.971884e+09,2676.112316,0.719299,5.044452e-16,0.122892,1.647250,-0.901029,2.371184,...,0,650271,1.778989e-08,0.086148,0.086148,0.128280,0.095920,0.163442,0.160894,0.818947
70,3.885330e-09,2.861043e+09,6.573347e+09,4286.088691,1.363366,3.503633e-16,0.021737,5.747365,-0.658114,0.111361,...,0,847151,1.435195e-08,0.067102,0.067102,0.102981,0.075244,0.134587,0.132137,0.636195
7,3.882350e-09,2.114953e+09,5.247048e+09,800.426721,0.187005,7.160515e-16,0.034283,3.727643,-0.314831,2.355300,...,0,242881,2.935416e-08,0.067075,0.067075,0.100310,0.074755,0.128012,0.126412,0.464068


In [11]:
# # 10. Final cleanup
# restore_pulsars()
# print("Restored all live pulsars to their original TOAs.")
